# FreshGuard v2 - 04 Train Classifier DINOv3

Trains DINOv3-S/16 for the unchanged 24-class freshness contract. LoRA adapters are applied inside this notebook and merged into the exported PyTorch state dict so the Streamlit runtime needs only `torch` and `timm`.

Output artifact: `/kaggle/working/freshguard_v2_classifier_artifacts/dinov3_vits16_food_freshness_v2.pt`.

In [ ]:
from __future__ import annotations

import json
import math
import os
from pathlib import Path

import numpy as np
import pandas as pd
import timm
import torch
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms

KAGGLE_INPUT = Path('/kaggle/input')
MANIFEST_PATH = next(iter(sorted(KAGGLE_INPUT.rglob('classifier_manifest.csv'))), None)
if MANIFEST_PATH is None:
    raise FileNotFoundError('Attach notebook 01 output as freshguard-v2-splits.')
SPLITS_DIR = MANIFEST_PATH.parent
OUT_DIR = Path('/kaggle/working/freshguard_v2_classifier_artifacts')
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = 'vit_small_patch16_dinov3'
IMAGE_SIZE = 256
HEAD_HIDDEN_DIM = 512
BATCH_SIZE = int(os.environ.get('CLS_BATCH', '64'))
EPOCHS = int(os.environ.get('CLS_EPOCHS', '12'))
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EXPECTED_MODEL_DEVICE = torch.device('cuda:0') if DEVICE.type == 'cuda' else DEVICE
USE_AMP = DEVICE.type == 'cuda'

PRODUCE_TYPES = ('apple', 'banana', 'bellpepper', 'bitter_gourd', 'carrot', 'cucumber', 'mango', 'okra', 'orange', 'potato', 'strawberry', 'tomato')
FRESHNESS_LEVELS = ('fresh', 'rotten')
CLASS_NAMES = tuple(f'{produce}_{freshness}' for produce in PRODUCE_TYPES for freshness in FRESHNESS_LEVELS)
CLASS_TO_INDEX = {name: i for i, name in enumerate(CLASS_NAMES)}

manifest = pd.read_csv(MANIFEST_PATH)
manifest = manifest[manifest['combined_label'].isin(CLASS_TO_INDEX)].copy()
print({'rows': len(manifest), 'splits': manifest['split'].value_counts().to_dict(), 'device': str(DEVICE), 'cuda_devices': torch.cuda.device_count()})

In [ ]:
class FreshnessDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, transform: transforms.Compose) -> None:
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        row = self.frame.iloc[index]
        image = Image.open(row.path).convert('RGB')
        label = CLASS_TO_INDEX[row.combined_label]
        return self.transform(image), torch.tensor(label, dtype=torch.long)

train_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5, hue=0.05),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

train_df = manifest[manifest['split'] == 'train'].copy()
val_df = manifest[manifest['split'] == 'val'].copy()
test_df = manifest[manifest['split'] == 'test'].copy()
class_counts = train_df['combined_label'].value_counts().to_dict()
def sample_weight(row: pd.Series) -> float:
    base = 1.0 / class_counts[row['combined_label']]
    if row.get('supervision') == 'fresh_assumed':
        return base * 0.25
    return base

weights = train_df.apply(sample_weight, axis=1).to_numpy()
sampler = WeightedRandomSampler(weights=torch.DoubleTensor(weights), num_samples=len(weights), replacement=True)

train_loader = DataLoader(FreshnessDataset(train_df, train_tf), batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True)
val_loader = DataLoader(FreshnessDataset(val_df, eval_tf), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(FreshnessDataset(test_df, eval_tf), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, rank: int = 16, alpha: int = 16) -> None:
        super().__init__()
        self.base = base
        self.rank = rank
        self.alpha = alpha
        self.lora_a = nn.Linear(base.in_features, rank, bias=False)
        self.lora_b = nn.Linear(rank, base.out_features, bias=False)
        nn.init.kaiming_uniform_(self.lora_a.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_b.weight)
        for parameter in self.base.parameters():
            parameter.requires_grad = False

    def forward(self, tensor: torch.Tensor) -> torch.Tensor:
        return self.base(tensor) + self.lora_b(self.lora_a(tensor)) * (self.alpha / self.rank)

    def merged(self) -> nn.Linear:
        merged = nn.Linear(self.base.in_features, self.base.out_features, bias=self.base.bias is not None)
        merged.weight.data.copy_(self.base.weight.data + (self.lora_b.weight.data @ self.lora_a.weight.data) * (self.alpha / self.rank))
        if self.base.bias is not None:
            merged.bias.data.copy_(self.base.bias.data)
        return merged

class DinoV3FreshnessModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.backbone = timm.create_model(MODEL_NAME, pretrained=True, num_classes=0)
        self.head = nn.Sequential(
            nn.Linear(int(self.backbone.num_features), HEAD_HIDDEN_DIM),
            nn.GELU(),
            nn.Dropout(p=0.1),
            nn.Linear(HEAD_HIDDEN_DIM, len(CLASS_NAMES)),
        )

    def forward(self, tensor: torch.Tensor) -> torch.Tensor:
        return self.head(self.backbone(tensor))

def apply_lora_to_last_blocks(model: DinoV3FreshnessModel, rank: int = 16) -> None:
    for parameter in model.backbone.parameters():
        parameter.requires_grad = False
    for block in model.backbone.blocks[-2:]:
        for name in ('qkv', 'proj'):
            module = getattr(block.attn, name, None)
            if isinstance(module, nn.Linear):
                setattr(block.attn, name, LoRALinear(module, rank=rank, alpha=rank))

def merge_lora_modules(module: nn.Module) -> None:
    for name, child in list(module.named_children()):
        if isinstance(child, LoRALinear):
            setattr(module, name, child.merged())
        else:
            merge_lora_modules(child)

base_model = DinoV3FreshnessModel().to(DEVICE)
apply_lora_to_last_blocks(base_model, rank=16)
base_model = base_model.to(DEVICE)
bad_devices = [(name, str(param.device)) for name, param in base_model.named_parameters() if param.device != EXPECTED_MODEL_DEVICE]
if bad_devices:
    raise RuntimeError(f'Model parameters not on {EXPECTED_MODEL_DEVICE}: {bad_devices[:10]}')
trainable = [p for p in base_model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW([
    {'params': base_model.head.parameters(), 'lr': 3e-4},
    {'params': [p for n, p in base_model.named_parameters() if p.requires_grad and not n.startswith('head.')], 'lr': 3e-5},
], weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss()
model: nn.Module = base_model
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(base_model)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
print({'trainable_params': sum(p.numel() for p in trainable), 'data_parallel': isinstance(model, nn.DataParallel), 'amp': USE_AMP})

In [ ]:
def run_eval(loader: DataLoader) -> dict[str, float]:
    model.eval()
    y_true: list[int] = []
    y_pred: list[int] = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                logits = model(images)
            preds = logits.argmax(dim=1).cpu().tolist()
            y_pred.extend(preds)
            y_true.extend(labels.tolist())
    return {
        'top1_accuracy': float(accuracy_score(y_true, y_pred)),
        'macro_f1': float(f1_score(y_true, y_pred, average='macro')),
    }

best_macro_f1 = -1.0
best_path = OUT_DIR / 'best_classifier_state.pt'
history: list[dict[str, float]] = []
for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            loss = criterion(model(images), labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += float(loss.item()) * images.size(0)
    scheduler.step()
    metrics = run_eval(val_loader)
    metrics['epoch'] = float(epoch)
    metrics['train_loss'] = running_loss / max(1, len(train_df))
    history.append(metrics)
    print(metrics)
    if metrics['macro_f1'] > best_macro_f1:
        best_macro_f1 = metrics['macro_f1']
        torch.save(base_model.state_dict(), best_path)

(OUT_DIR / 'classifier_train_history.json').write_text(json.dumps(history, indent=2))

In [ ]:
base_model.load_state_dict(torch.load(best_path, map_location=DEVICE))
test_metrics = run_eval(test_loader)
merge_lora_modules(base_model)
base_model.eval()

artifact = OUT_DIR / 'dinov3_vits16_food_freshness_v2.pt'
checkpoint = {
    'model_state_dict': base_model.state_dict(),
    'class_names': CLASS_NAMES,
    'model_name': MODEL_NAME,
    'image_size': IMAGE_SIZE,
    'crop_pct': 1.0,
    'head_hidden_dim': HEAD_HIDDEN_DIM,
    'test_metrics': test_metrics,
    'note': 'LoRA adapters merged into backbone weights before export.',
}
torch.save(checkpoint, artifact)
summary = {'artifact': str(artifact), 'size_bytes': artifact.stat().st_size, **test_metrics}
(OUT_DIR / 'classifier_eval_summary.json').write_text(json.dumps(summary, indent=2))
print(summary)